# Введение в PyTorch

PyTorch — это библиотека для вычислений с поддержкой ускорителей (например, видеокарт) и встроенной системой автоматического дифференцирования. Её можно рассматривать как удобную среду для дифференцируемого программирования.

В отличие от традиционных библиотек линейной алгебры, PyTorch не просто выполняет численные операции — он строит вычислительный граф заданной сложной функции и с его помощью автоматически вычисляет производные.

### Автоматические дифференцирование

Автоматическое дифференцирование  — это метод вычисления точных производных сложных функций, заданных программой. Важно:

1. Это не численное дифференцирование, т.е. не конечные разности.

2. Это не символьное дифференцирование, как в Wolfram.

Это алгоритмическое применение правила вычисления производной сложной функции к вычислительному графу.


### Почему PyTorch?
Потому что это де-факто стандарт в области обучения нейронных сетей.

#### paperswithcode.com:
![](https://cdn.prod.website-files.com/68ed36e99e31581dedf5dcb1/68ed36e99e31581dedf5f298_668edbcaa662eb1dd9cf12d6_7zVSipS.webp)

#### Репозитории:
![](https://cdn.prod.website-files.com/67a1e6de2f2eab2e125f8b9a/67be0ec8aef316bdeea12871_percentage_repo_2023.png)




## 1. Тензоры в PyTorch

Тензором в PyTorch называется основной объект вычислений, задающий многомерный массив.

**Пример 1.1** Создайте tensors тремя способами: из списка Python, из массива NumPy и с помощью `torch.zeros` формы (3, 4). Выведите их shape и dtype.

**Пример 1.2** Создайте tensor формы (2, 3), перенесите на GPU при наличии (иначе оставьте на CPU), выполните поэлементное умножение (element-wise) с tensor той же формы. Выведите результат и device.

In [1]:
import torch
import numpy as np

# 1.1
t_from_list = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
arr = np.array([[5, 6], [7, 8]], dtype=np.float32)
t_from_numpy = torch.from_numpy(arr)
t_zeros = torch.zeros(3, 4)

for name, t in [("from_list", t_from_list), ("from_numpy", t_from_numpy), ("zeros", t_zeros)]:
    print(f"{name}: shape={t.shape}, dtype={t.dtype}")

# 1.2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
a = torch.randn(2, 3, device=device)
b = torch.randn(2, 3, device=device)
c = a * b
print(f"\nElement-wise product, device: {c.device}")
print(c)

from_list: shape=torch.Size([2, 2]), dtype=torch.float32
from_numpy: shape=torch.Size([2, 2]), dtype=torch.float32
zeros: shape=torch.Size([3, 4]), dtype=torch.float32

Element-wise product, device: cpu
tensor([[-4.8869, -0.0874, -0.6309],
        [ 0.3201, -0.0636,  0.5108]])


**Задача 1.1** Создайте тензор формы (4, 5) с помощью `torch.ones`, затем создайте тензор той же формы из списка списков (любые числа). Вычислите их поэлементную разность и выведите результат, а также проверку `[result].dtype`.

In [5]:
a = torch.ones(2,2)
b = torch.tensor([[2,-3],[3,5]])
dif = a - b
print(dif)
print(c.dtype)

tensor([[-1.,  4.],
        [-2., -4.]])
torch.float32


## 2. Autograd
В PyTorch автоматическое дифференцирование реализовано механизмом autograd, который динамически строит вычислительный граф и применяет правило дифференцирования производных сложных функций. Если тензор создан с параметром requires_grad=True, все операции над ним начинают отслеживаться: результат каждой операции хранит информацию о том, каким отображением он получен. Тем самым программа не просто вычисляет значение функции, а фактически задаёт композицию элементарных отображений.

При вызове метода .backward() запускается обратный проход по графу (reverse-mode AD).

Таким образом, autograd можно рассматривать как практическую реализацию оператора дифференцирования для функций, заданных алгоритмически: программа определяет композицию отображений, а система автоматически вычисляет её дифференциал.


**Пример 2.1** Задайте скалярный loss как функцию от 1D tensor `x` (например, сумма квадратов). Включите градиенты для `x`, вычислите loss, вызовите `.backward()` и выведите `x.grad`. Проверьте, что gradient совпадает с 2*x аналитически.

**Пример 2.2** Вычислите gradient выражения `z = sum(x @ y)` по `x`, где `x` имеет shape (2, 3), а `y` — (3, 1). Выведите shape `x.grad`.

**Пример 2.3** Визуализируйте граф вычислений с помощью torchviz

In [10]:
# 2.1
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
loss = (x ** 2).sum()
loss.backward()
print("x.grad (d/dx sum(x^2) = 2*x):", x.grad)
print("2*x:", 2 * x.detach())

x.grad (d/dx sum(x^2) = 2*x): tensor([2., 4., 6.])
2*x: tensor([2., 4., 6.])


In [17]:
# 2.2
x = torch.randn(2, 3, requires_grad=True)
y = torch.randn(3, 1, requires_grad=True)
z = x @ y
z = sum(z)
z.backward()
print("\nx.shape:", x.shape, "x.grad.shape:", x.grad.shape)
print("Gradient по x имеет ту же shape, что и x.")


x.shape: torch.Size([2, 3]) x.grad.shape: torch.Size([2, 3])

x.shape: torch.Size([3, 1]) x.grad.shape: torch.Size([3, 1])
Gradient по x имеет ту же shape, что и x.


In [21]:
# 2.3
from torchviz import make_dot

dot = make_dot(z, params={"x": x, "y": y})
dot.render("z_graph", format="png")

'z_graph.png'

**Задача 2.1** Задайте два 1D тензора `a` и `b` длины 4 с `requires_grad=True`. Вычислите скаляр `loss = (a * b).sum()`, вызовите `loss.backward()` и выведите `a.grad` и `b.grad`. Проверьте аналитически: для loss = sum(a_i * b_i) градиент по a есть b, по b -- a.

In [24]:
a = torch.randn(4, requires_grad=True)
b = torch.randn(4, requires_grad=True)
loss = (a * b).sum()
loss.backward()
print("a.grad:", a.grad, b == a.grad)
print("b.grad:", b.grad, a == b.grad)


a.grad: tensor([ 1.2567,  0.8266, -0.9255, -0.5241]) tensor([True, True, True, True])
b.grad: tensor([ 0.6307, -0.8112, -1.1162, -0.7193]) tensor([True, True, True, True])


## 3. nn.Module

В PyTorch nn.Module — это базовый класс для построения моделей как программных объектов. Хотя вычислительный граф можно строить напрямую, оперируя тензорами и autograd, использование nn.Module даёт существенные преимущества на уровне архитектуры кода.

Во-первых, модуль автоматически регистрирует обучаемые параметры (nn.Parameter) и вложенные подмодули. Это избавляет от ручного управления списками параметров и гарантирует, что оптимизатор «видит» всё, что должно обновляться.

Во-вторых, nn.Module инкапсулирует вычисление в методе forward, превращая модель в обычный вызываемый объект. Это делает код декларативным: архитектура описывается структурой класса, а не процедурной сборкой графа на каждом шаге.

В-третьих, модуль обеспечивает:

1. рекурсивный перенос параметров на устройство (.to(device)),

2. переключение режимов обучения и инференса (.train() / .eval()),

3. сохранение и загрузку состояния (state_dict()),

4. корректную композицию сложных моделей из более простых.

**Пример 3.1** Определите класс `SmallMLP(nn.Module)` с одним скрытым слоем: input 10, hidden 32, output 3. Реализуйте `forward(self, x)` и выведите shape выхода для batch из 5 примеров.

**Пример 3.2** Подсчитайте число trainable parameters в модели и выведите имена слоёв и shape их параметров.

In [55]:
import torch.nn as nn

class SmallMLP(nn.Module):
    def __init__(self, in_dim=10, hidden=32, out_dim=3):
        super().__init__()
        self.lin1 = nn.Linear(in_dim, hidden)
        self.lin2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        x = torch.relu(self.lin1(x))
        return self.lin2(x)

model = SmallMLP()
x = torch.randn(5, 10)
out = model(x)
print("Output shape (batch=5, out_dim=3):", out.shape)

total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Trainable parameters (всего):", total)
for name, p in model.named_parameters():
    print(f"  {name}: {p.shape}")

Output shape (batch=5, out_dim=3): torch.Size([5, 3])
Trainable parameters (всего): 451
  lin1.weight: torch.Size([32, 10])
  lin1.bias: torch.Size([32])
  lin2.weight: torch.Size([3, 32])
  lin2.bias: torch.Size([3])


**Задача 3.1** Определите класс `TwoLayerNet(nn.Module)` с двумя линейными слоями (input 8, hidden 16, output 4) и ReLU между ними. Создайте модель, подайте батч формы (10, 8) и выведите shape выхода. Подсчитайте число параметров.

In [28]:
class TwoLayerNet(nn.Module):
    def __init__(self, in_dim=8, hidden=16, out_dim=4):
      super().__init__()
      self.lin1 = nn.Linear(in_dim, hidden)
      self.lin2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
      x = torch.relu(self.lin1(x))
      return self.lin2(x)
x = torch.randn(10, 8)
TwoLayerNet = TwoLayerNet()
out = TwoLayerNet(x)
print("Output shape (batch=10, out_dim=4):", out.shape)
total = sum(p.numel() for p in TwoLayerNet.parameters() if p.requires_grad)
print(total)

Output shape (batch=10, out_dim=4): torch.Size([10, 4])
212


## 4. Loss, optimizer и шаг обучения

В PyTorch процесс обучения модели структурируется вокруг трёх компонентов: функции потерь (loss), оптимизатора (optimizer) и шага обучения.

С программной точки зрения loss это обычный вызываемый объект (например, nn.MSELoss, nn.CrossEntropyLoss), возвращающий тензор-скаляр, по которому затем вызывается .backward().

Оптимизатор (например, torch.optim.SGD, Adam) — это объект, который:

1. получает список параметров модели,

2. использует значения их .grad,

3. обновляет параметры по заданному правилу.

Важно, что оптимизатор не вычисляет градиенты — он лишь применяет шаг обновления, предполагая, что autograd уже заполнил .grad.

**Пример 4.1** Для приведённого выше `SmallMLP` создайте dummy dataset: входы формы (N, 10) и целочисленные метки из {0, 1, 2}. Используйте CrossEntropyLoss и SGD optimizer.

**Пример 4.2** Реализуйте одну training epoch: обнуление градиентов, forward pass, loss backward, optimizer step. Выполните 3 epoch и выводите loss на каждой epoch.

In [56]:
N = 100
X = torch.randn(N, 10)
y = torch.randint(0, 3, (N,))

model = SmallMLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

for epoch in range(3):
    model.train()
    optimizer.zero_grad()
    logits = model(X)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}, loss = {loss.item():.4f}")

Epoch 1, loss = 1.0945
Epoch 2, loss = 1.0945
Epoch 3, loss = 1.0944


**Задача 4.1** Для `SmallMLP` создайте синтетические данные: 50 примеров, вход (50, 10), метки из {0, 1, 2}. Обучите модель 10 эпох с `nn.CrossEntropyLoss` и `torch.optim.Adam(lr=0.01)`. Выводите loss на каждой эпохе.

In [30]:
N = 50
x = torch.randn(N, 10)
y = torch.randint(0, 3, (N,))

model = SmallMLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(100):
    model.train()
    optimizer.zero_grad()
    logits = model(x)
    loss = criterion(logits, y)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}, loss = {loss.item():.4f}")

Epoch 1, loss = 1.2183
Epoch 2, loss = 1.1664
Epoch 3, loss = 1.1237
Epoch 4, loss = 1.0881
Epoch 5, loss = 1.0576
Epoch 6, loss = 1.0304
Epoch 7, loss = 1.0055
Epoch 8, loss = 0.9818
Epoch 9, loss = 0.9588
Epoch 10, loss = 0.9358
Epoch 11, loss = 0.9130
Epoch 12, loss = 0.8893
Epoch 13, loss = 0.8655
Epoch 14, loss = 0.8417
Epoch 15, loss = 0.8176
Epoch 16, loss = 0.7932
Epoch 17, loss = 0.7687
Epoch 18, loss = 0.7438
Epoch 19, loss = 0.7189
Epoch 20, loss = 0.6948
Epoch 21, loss = 0.6711
Epoch 22, loss = 0.6475
Epoch 23, loss = 0.6239
Epoch 24, loss = 0.6008
Epoch 25, loss = 0.5787
Epoch 26, loss = 0.5574
Epoch 27, loss = 0.5362
Epoch 28, loss = 0.5154
Epoch 29, loss = 0.4957
Epoch 30, loss = 0.4766
Epoch 31, loss = 0.4580
Epoch 32, loss = 0.4398
Epoch 33, loss = 0.4224
Epoch 34, loss = 0.4049
Epoch 35, loss = 0.3879
Epoch 36, loss = 0.3712
Epoch 37, loss = 0.3549
Epoch 38, loss = 0.3387
Epoch 39, loss = 0.3230
Epoch 40, loss = 0.3076
Epoch 41, loss = 0.2924
Epoch 42, loss = 0.2778
E

## 5. DataLoader и Dataset

В PyTorch работа с данными при обучении строится на двух абстракциях: **Dataset** и **DataLoader**.

**Dataset** — это объект, представляющий набор данных. От пользователя требуется реализовать подкласс `torch.utils.data.Dataset` с двумя методами: `__len__()` возвращает число примеров в наборе, `__getitem__(idx)` возвращает пару (вход, метка) для заданного индекса. Такой интерфейс позволяет единообразно описывать данные любого происхождения: тензоры в памяти, файлы на диске, выборки из базы. Код обучения тогда не зависит от способа хранения данных.

**DataLoader** — обёртка над Dataset, которая формирует батчи для обучения. Он принимает dataset, размер батча (`batch_size`), опцию перемешивания (`shuffle`) и при необходимости число потоков загрузки (`num_workers`). При итерации DataLoader выдаёт батчи тензоров (входы и метки), уже собранные в один тензор по первому измерению. Это избавляет от ручного разбиения на батчи и упрощает цикл по эпохам.

**Пример 5.1** Создайте подкласс `torch.utils.data.Dataset` для тех же данных (X, y): реализуйте `__len__` и `__getitem__(idx)`.

**Пример 5.2** Создайте `DataLoader` с batch_size=16 и shuffle=True. Возьмите один batch и выведите shape входов и меток в batch.

In [57]:
from torch.utils.data import Dataset, DataLoader

class DummyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

ds = DummyDataset(X, y)
loader = DataLoader(ds, batch_size=16, shuffle=True)

batch_X, batch_y = next(iter(loader))
print("Batch X, shape:", batch_X.shape)
print("Batch y, shape:", batch_y.shape)

Batch X, shape: torch.Size([16, 10])
Batch y, shape: torch.Size([16])


**Задача 5.1** Реализуйте класс `TensorDataset` (подкласс `torch.utils.data.Dataset`) для пар тензоров `X` формы (N, 6) и `y` формы (N,). Создайте DataLoader с `batch_size=8`, `shuffle=False`. Пройдите один полный цикл по loader и выведите количество батчей и shape последнего батча (если последний батч неполный — shape всё равно должен быть корректным).

In [34]:
class CustomTensorDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

N = 100
X = torch.randn(N, 6)
y = torch.randint(0, 2, (N,))

dataset = CustomTensorDataset(X, y)
batch_size = 8
loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

batches_count = 0
last_batch_X_shape = None

for batch_X, batch_y in loader:
    batches_count += 1
    last_batch_X_shape = batch_X.shape

print(f"Общее количество объектов: {N}")
print(f"Размер батча: {batch_size}")
print(f"---")
print(f"Количество батчей: {batches_count}")
print(f"Shape последнего батча: {last_batch_X_shape}")

Общее количество объектов: 100
Размер батча: 8
---
Количество батчей: 13
Shape последнего батча: torch.Size([4, 6])


## 6. Полный training loop с DataLoader

**Пример 6.1** Обучите `SmallMLP` 5 epochs с помощью DataLoader. На каждой epoch итерируйте по batches, накапливайте loss и делайте optimizer step. Выводите средний loss за epoch.

In [58]:
model = SmallMLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(5):
    model.train()
    total_loss = 0.0
    n_batches = 0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    mean_loss = total_loss / n_batches
    print(f"Epoch {epoch + 1}, mean loss = {mean_loss:.4f}")

Epoch 1, mean loss = 1.1304
Epoch 2, mean loss = 1.0361
Epoch 3, mean loss = 0.9923
Epoch 4, mean loss = 0.9212
Epoch 5, mean loss = 0.8769


**Задача 6.1** Обучите `SmallMLP` на данных из раздела 5 (DummyDataset + DataLoader с batch_size=16) 3 эпохи. Используйте Adam и CrossEntropyLoss. Выводите средний loss по эпохе и количество батчей за эпоху.

In [60]:
dataset = DummyDataset(X, y)
loader = DataLoader(dataset, batch_size=16)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
model = SmallMLP()
total_loss = 0.0
n_batches = 0

for epoch in range(3):
    total_loss = 0.0
    n_batches = 0
    model.train()
    for batch_X, batch_y in loader:
      optimizer.zero_grad()
      logits = model(batch_X)
      loss = criterion(logits, batch_y)
      loss.backward()
      optimizer.step()
      total_loss += loss.item()
      n_batches += 1
    mean_loss = total_loss / n_batches
    print(f"Epoch {epoch + 1}, mean loss = {mean_loss:.4f}")


Epoch 1, mean loss = 1.1278
Epoch 2, mean loss = 1.1278
Epoch 3, mean loss = 1.1278


## 7. torch.no_grad() и model.eval()

При валидации и инференсе градиенты не нужны, а хранение вычислительного графа для backward занимает память и время. Контекстный менеджер **`torch.no_grad()`** отключает запись операций для autograd: все вычисления внутри блока `with torch.no_grad():` не участвуют в построении графа. Это экономит память и ускоряет проход по валидационной выборке.

**`model.eval()`** переводит модель в режим инференса. Для слоёв вроде `Dropout` и `BatchNorm` поведение в обучении и при оценке различается: в train они используют текущий батч (dropout выключает часть нейронов, batchnorm обновляет статистики), в eval — фиксированное детерминированное поведение (dropout отключен, batchnorm использует накопленные среднее и дисперсию). Без вызова `model.eval()` метрики на валидации могут быть некорректными и невоспроизводимыми.

**Пример 7.1** Разбейте данные на train/val (например, 80/20). В каждом epoch после обучения на train пройдите по val в блоке `with torch.no_grad():`, переключив модель в `model.eval()`. Выводите train loss и validation loss (и при желании accuracy).

**Пример 7.2** Поясните, зачем нужны `model.eval()` и `no_grad` при валидации (в комментарии или print).

In [61]:
n_val = 20
X_train, X_val = X[:-n_val], X[-n_val:]
y_train, y_val = y[:-n_val], y[-n_val:]

model = SmallMLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(3):
    model.train()
    optimizer.zero_grad()
    logits_train = model(X_train)
    loss_train = criterion(logits_train, y_train)
    loss_train.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        logits_val = model(X_val)
        loss_val = criterion(logits_val, y_val)
        pred = logits_val.argmax(dim=1)
        acc = (pred == y_val).float().mean().item()
    print(f"Epoch {epoch + 1}: train loss = {loss_train.item():.4f}, val loss = {loss_val.item():.4f}, val acc = {acc:.4f}")

Epoch 1: train loss = 1.1806, val loss = 1.1507, val acc = 0.3000
Epoch 2: train loss = 1.1395, val loss = 1.1613, val acc = 0.2000
Epoch 3: train loss = 1.1047, val loss = 1.1732, val acc = 0.2000


**Задача 7.1** Разбейте данные (X, y) на train 70% и val 30% (без перемешивания индексов). Обучите модель 5 эпох: на train с градиентами, на val — в `torch.no_grad()` и `model.eval()`. Выводите train loss и val accuracy после каждой эпохи.

In [62]:
n_val = 30
X_train, X_val = X[:-n_val], X[-n_val:]
y_train, y_val = y[:-n_val], y[-n_val:]

model = SmallMLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)

for epoch in range(5):
    model.train()
    optimizer.zero_grad()
    logits_train = model(X_train)
    loss_train = criterion(logits_train, y_train)
    loss_train.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        logits_val = model(X_val)
        loss_val = criterion(logits_val, y_val)
        pred = logits_val.argmax(dim=1)
        acc = (pred == y_val).float().mean().item()
    print(f"Epoch {epoch + 1}: train loss = {loss_train.item():.4f}, val loss = {loss_val.item():.4f}, val acc = {acc:.4f}")

Epoch 1: train loss = 1.1037, val loss = 1.0994, val acc = 0.3667
Epoch 2: train loss = 1.0651, val loss = 1.1079, val acc = 0.3333
Epoch 3: train loss = 1.0302, val loss = 1.1191, val acc = 0.3000
Epoch 4: train loss = 0.9975, val loss = 1.1327, val acc = 0.3667
Epoch 5: train loss = 0.9665, val loss = 1.1488, val acc = 0.4667


## 8. Сохранение и загрузка модели

Параметры модели (и при необходимости буферы вроде running mean в BatchNorm) собираются в словарь методом **`model.state_dict()`**. Его можно сохранить в файл через `torch.save()` и позже загрузить в новый экземпляр той же архитектуры с помощью **`load_state_dict()`**. Состояние оптимизатора сохраняется так же — через `optimizer.state_dict()` — если нужно продолжить обучение с того же шага (learning rate schedule, моменты в Adam и т.д.).

**Пример 8.1** Сохраните state dict обученной модели в файл (например, `model.pt`) и state optimizer — в другой файл.

**Пример 8.2** Создайте новый экземпляр модели, загрузите state dict из файла и проверьте, что предсказания на фиксированном входе совпадают с исходной моделью.

In [63]:
PATH = "model.pt"
OPT_PATH = "optimizer.pt"

torch.save(model.state_dict(), PATH)
torch.save(optimizer.state_dict(), OPT_PATH)

model_loaded = SmallMLP()
model_loaded.load_state_dict(torch.load(PATH))
model_loaded.eval()

with torch.no_grad():
    test_input = torch.randn(2, 10)
    out_orig = model(test_input)
    out_loaded = model_loaded(test_input)
    match = torch.allclose(out_orig, out_loaded)
print("Predictions match after load:", match)

Predictions match after load: True


**Задача 8.1** Сохраните обученную модель в файл `my_model.pt`. Загрузите веса в новый экземпляр `SmallMLP` и убедитесь, что на одном и том же входе (например, `torch.randn(1, 10)`) предсказания совпадают (используйте `torch.allclose`).

In [64]:
PATH = "my_model.pt"
model = SmallMLP(in_dim=10, hidden=32, out_dim=3)
torch.save(model.state_dict(), PATH)

model_loaded = SmallMLP(in_dim=10, hidden=32, out_dim=3)
model_loaded.load_state_dict(torch.load(PATH, weights_only=True))

model.eval()
model_loaded.eval()

with torch.no_grad():
    test_input = torch.randn(1, 10)
    out_orig = model(test_input)
    out_loaded = model_loaded(test_input)
    match = torch.allclose(out_orig, out_loaded)

print(f"Совпадают ли предсказания? {match}")

Совпадают ли предсказания? True


## 9. Кастомный loss

Функция потерь в цикле обучения — это просто скаляр от тензоров с `requires_grad=True`; для backward достаточно, чтобы она была дифференцируемой. **Кастомный loss** можно задать функцией `(logits, targets) -> scalar` или классом-наследником `nn.Module` с методом `forward(logits, targets)`. Второй вариант удобен, когда у loss есть гиперпараметры (например, smoothing), которые нужно хранить в объекте.

**Пример 9.1** Добавьте к CrossEntropyLoss ручную L2-регуляризацию: loss_total = loss_ce + lambda * sum(w^2) по всем весам модели. Используйте один и тот же optimizer без weight_decay.

**Пример 9.2** Реализуйте кастомный loss как класс (наследник nn.Module) или функцию, принимающую logits и targets, и используйте его в цикле обучения вместо CrossEntropyLoss (например, MSE для регрессии или свой вариант).

In [ ]:
def loss_ce_with_l2(model, logits, targets, l2_lambda=0.01):
    ce = nn.functional.cross_entropy(logits, targets)
    l2 = sum((p ** 2).sum() for p in model.parameters())
    return ce + l2_lambda * l2

model = SmallMLP()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(3):
    model.train()
    optimizer.zero_grad()
    logits = model(X)
    loss = loss_ce_with_l2(model, logits, y)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch + 1}, loss (CE + L2) = {loss.item():.4f}")

# 9.2: custom loss as nn.Module (e.g. smoothed cross-entropy)
class LabelSmoothingLoss(nn.Module):
    def __init__(self, n_classes=3, smoothing=0.1):
        super().__init__()
        self.n_classes = n_classes
        self.smoothing = smoothing

    def forward(self, logits, targets):
        log_probs = torch.log_softmax(logits, dim=1)
        one_hot = torch.zeros_like(logits).scatter_(1, targets.unsqueeze(1), 1.0)
        smooth = one_hot * (1 - self.smoothing) + self.smoothing / self.n_classes
        return -(smooth * log_probs).sum(dim=1).mean()

criterion_custom = LabelSmoothingLoss(n_classes=3, smoothing=0.1)
loss_val = criterion_custom(model(X), y)
print("\nCustom loss (label smoothing) sample value:", loss_val.item())

**Задача 9.1** Реализуйте функцию или класс `FocalLoss` (или упрощённый вариант: взвешенный CrossEntropy с весами по классам). Используйте его в цикле обучения вместо обычного CrossEntropyLoss на 2–3 эпохи и выведите итоговый loss.

In [66]:
class SimpleFocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(logits, targets, weight=self.weight, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma * ce_loss).mean()
        return focal_loss

model = SmallMLP(in_dim=6, out_dim=3)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = SimpleFocalLoss(gamma=2.0, weight=torch.tensor([1.0, 2.0, 1.0]))

X = torch.randn(100, 6)
y = torch.randint(0, 3, (100,))

for epoch in range(3):
    model.train()
    optimizer.zero_grad()

    logits = model(X)
    loss = criterion(logits, y)

    loss.backward()
    optimizer.step()

    print(f"Epoch {epoch + 1}, Focal Loss = {loss.item():.4f}")

Epoch 1, Focal Loss = 0.9040
Epoch 2, Focal Loss = 0.8398
Epoch 3, Focal Loss = 0.7941


## 10. Кастомная autograd.Function

Когда нужно встроить в вычислительный граф операцию, для которой PyTorch не умеет сам считать градиент (или её вообще нет в библиотеке), используют **`torch.autograd.Function`**. Подкласс задаёт статические методы `forward(ctx, ...)` и `backward(ctx, grad_output)`: в forward выполняются вычисления, при этом через `ctx.save_for_backward(...)` можно сохранить тензоры, нужные в backward; в backward по правилу цепи возвращается градиент по каждому входу. Вызов в коде идёт через `YourFunction.apply(inputs)`, так что операция ведёт себя как обычный узел графа.

**Пример 10.1** Реализуйте свою функцию `MySquare` как подкласс `torch.autograd.Function`: в `forward(ctx, x)` верните `x * x`, в `backward(ctx, grad_output)` верните градиент по входу (для y = x^2 это 2*x * grad_output). Проверьте на tensor с requires_grad=True, что backward даёт ожидаемый градиент.

**Пример 10.2** Используйте `MySquare.apply(x)` в цепочке вычислений (например, loss = MySquare.apply(x).sum()) и убедитесь, что .backward() работает.

In [ ]:
from torch.autograd import Function

class MySquare(Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x * x

    @staticmethod
    def backward(ctx, grad_output):
        (x,) = ctx.saved_tensors
        return 2 * x * grad_output

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = MySquare.apply(x)
loss = y.sum()
loss.backward()
print("x.grad (d/dx sum(x^2) = 2*x):", x.grad)
print("Expected 2*x:", 2 * x.detach())

**Задача 10.1** Реализуйте свою autograd-функцию `MyReLU`: в `forward` верните `torch.clamp(x, min=0)`, в `backward` верните `grad_output` там, где `x > 0`, и 0 иначе. Проверьте на тензоре с `requires_grad=True`, что градиенты совпадают с ожидаемыми (сравните с `torch.relu(x)` и его градиентом).

In [70]:
import torch
from torch.autograd import Function

class MyReLU(Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return torch.clamp(x, min=0)

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        grad_input = grad_output.clone()
        grad_input[x <= 0] = 0
        return grad_input
x = torch.tensor([-2.0, 0.0, 3.0, -0.5, 1.0], requires_grad=True)

y_custom = MyReLU.apply(x)
loss_custom = y_custom.sum()
loss_custom.backward()
grad_custom = x.grad.clone()

x.grad.zero_()

y_torch = torch.relu(x)
loss_torch = y_torch.sum()
loss_torch.backward()
grad_torch = x.grad

print(f"Входной тензор x: {x.detach()}")
print(f"Градиент MyReLU:  {grad_custom}")
print(f"Градиент torch.relu: {grad_torch}")

Входной тензор x: tensor([-2.0000,  0.0000,  3.0000, -0.5000,  1.0000])
Градиент MyReLU:  tensor([0., 0., 1., 0., 1.])
Градиент torch.relu: tensor([0., 0., 1., 0., 1.])


**Задача 10.2** Постройте вычислительный граф из двух кастомных autograd-функций с кастомными градиентами (например, цепочка из двух своих `Function`: первая считает что-то в `forward`/`backward`, вторая — ещё одно преобразование). Подайте тензор с `requires_grad=True`, пройдите по графу и вызовите `.backward()`.

In [72]:
class MyCube(Function):
    @staticmethod
    def forward(ctx, x):
        ctx.save_for_backward(x)
        return x ** 3

    @staticmethod
    def backward(ctx, grad_output):
        x, = ctx.saved_tensors
        return grad_output * (3 * x**2)

class MyExp(Function):
    @staticmethod
    def forward(ctx, x):
        result = torch.exp(x)
        ctx.save_for_backward(result)
        return result

    @staticmethod
    def backward(ctx, grad_output):
        result, = ctx.saved_tensors
        return grad_output * result

x = torch.tensor([1.5], requires_grad=True)

cube_out = MyCube.apply(x)
exp_out = MyExp.apply(cube_out)
loss = exp_out.sum()

loss.backward()

expected_grad = torch.exp(x**3) * (3 * x**2)

print(f"Вход x: {x.item()}")
print(f"Результат f(x) = exp(x^3): {exp_out.item():.4f}")
print(f"Градиент (custom): {x.grad.item():.4f}")
print(f"Градиент (теоретический): {expected_grad.item():.4f}")

print(f"Совпадение: {torch.allclose(x.grad, expected_grad)}")

Вход x: 1.5
Результат f(x) = exp(x^3): 29.2243
Градиент (custom): 197.2639
Градиент (теоретический): 197.2639
Совпадение: True
